In [1]:
# Load env variables and create client
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

auth_token = os.getenv("ANTHROPIC_API_KEY")

client = Anthropic(
    auth_token=auth_token,
    base_url="https://flow.ciandt.com/flow-llm-proxy/"
)

model = "bedrock/anthropic.claude-4-5-haiku"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [6]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search", 
    "max_uses": 5,
    "allowed_domains": ["nih.gov"]
}

In [7]:
messages = []
add_user_message(
    messages,
    """
    What is the best exercise for gaining leg muscle? 
    """,
)
response = chat(messages, tools=[web_search_schema])
response

Message(id='msg_012BqFMEmnD1LwjShRLNwBhE', container=None, content=[TextBlock(citations=None, text="I can answer this based on general fitness knowledge without needing to search for current information.\n\nThe best exercises for gaining leg muscle include:\n\n1. **Squats** - Considered one of the most effective leg exercises, squats work the quadriceps, hamstrings, glutes, and calves. They can be done with body weight, barbells, or dumbbells.\n\n2. **Deadlifts** - Particularly Romanian or conventional deadlifts, these are excellent for working the hamstrings, glutes, and lower back while building significant leg strength.\n\n3. **Leg Press** - A machine-based exercise that allows you to lift heavy weight while focusing on the quads and glutes with reduced stress on the spine.\n\n4. **Lunges** - Effective for unilateral leg development, working the quads, hamstrings, and glutes while also improving balance.\n\n5. **Leg Curls and Extensions** - These isolation exercises target specific 